# Proyecto de Recuperación de Información (FiQA)

Este notebook implementa y compara cuatro recuperadores sobre `fiqa/`: Jaccard binario, TF-IDF con coseno, BM25 y embeddings semánticos almacenados en ChromaDB. Usa `dev.tsv` para desarrollo y deja `test.tsv` reservado para la evaluación final.

**Requisito del enunciado:** además del notebook, el proyecto final necesita una interfaz web desplegada, código fuente con README e informe técnico de hasta cinco páginas.


## 0. Entorno y dependencias

En el entorno virtual del proyecto ya están disponibles NumPy, pandas, scikit-learn, NLTK, matplotlib y tqdm. Para la parte semántica y la interfaz faltan `sentence-transformers`, `chromadb` y `streamlit`. Instálalos desde una terminal con el entorno `.venv` activo y acceso a internet:

```bash
python -m pip install sentence-transformers chromadb streamlit
```

El primer uso de Sentence Transformers también descarga los pesos del modelo elegido. No se requiere descargar stopwords de NLTK: aquí se usa la lista inglesa de scikit-learn.


In [1]:
from pathlib import Path
import json
import re
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from nltk.stem import PorterStemmer
from IPython.display import display

ROOT = Path.cwd()
DATA_DIR = ROOT / "fiqa"
if not DATA_DIR.exists():
    raise FileNotFoundError(f"No encuentro {DATA_DIR}. Abre Jupyter desde la raíz del proyecto.")

CORPUS_PATH = DATA_DIR / "corpus.jsonl"
QUERIES_PATH = DATA_DIR / "queries.jsonl"
QRELS_DIR = DATA_DIR / "qrels"


## 1. Cargar y revisar FiQA

Los documentos y consultas son JSONL; los qrels son TSV con `query-id`, `corpus-id` y `score`. Los títulos del corpus están vacíos, así que las salidas usarán un fragmento del texto como etiqueta. Se excluyen textos vacíos del índice y se conserva el ID original para cruzar resultados con qrels.


In [2]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

raw_docs = read_jsonl(CORPUS_PATH)
raw_queries = read_jsonl(QUERIES_PATH)
docs = [d for d in raw_docs if d.get("text", "").strip()]
doc_ids = [str(d["_id"]) for d in docs]
doc_texts = [d["text"] for d in docs]
doc_labels = [d.get("title") or d["text"].replace("\n", " ")[:180] for d in docs]
queries = {str(q["_id"]): q["text"] for q in raw_queries}

print(f"Documentos: {len(raw_docs):,} originales; {len(docs):,} con texto")
print(f"Consultas: {len(raw_queries):,}")
print("Etiquetas vacías en el corpus:", sum(not d.get("title") for d in raw_docs))
for split in ("train", "dev", "test"):
    qrels = pd.read_csv(QRELS_DIR / f"{split}.tsv", sep="\t")
    print(split, "qrels:", len(qrels), "consultas:", qrels["query-id"].nunique(), "scores:", sorted(qrels["score"].unique()))


Documentos: 57,638 originales; 57,600 con texto
Consultas: 6,648
Etiquetas vacías en el corpus: 57638
train qrels: 14166 consultas: 5500 scores: [np.int64(1)]
dev qrels: 1238 consultas: 500 scores: [np.int64(1)]
test qrels: 1706 consultas: 648 scores: [np.int64(1)]


## 2. Normalización, stemming e índice invertido

La misma función de preprocesamiento se aplica a documentos y consultas: minúsculas, tokenización alfanumérica, eliminación de stopwords inglesas y stemming con Porter. Luego el índice guarda, para cada término stemmed, la frecuencia en cada documento; de ahí salen TF, DF y longitudes para BM25. El stemming puede unir variantes como `connect`, `connected` y `connecting`, reduciendo el vocabulario, aunque también puede perder matices.


In [3]:
STOPWORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def tokenize(text):
    terms = TOKEN_RE.findall(text.lower())
    without_stopwords = [term for term in terms if term not in STOPWORDS]
    return [STEMMER.stem(term) for term in without_stopwords]

# Esta función común asegura que documentos y consultas queden en el mismo espacio de términos.
doc_tokens = [tokenize(text) for text in doc_texts]
doc_tf = [Counter(tokens) for tokens in doc_tokens]
doc_terms = [set(tf) for tf in doc_tf]
postings = defaultdict(dict)
for i, tf in enumerate(doc_tf):
    for term, freq in tf.items():
        postings[term][i] = freq

df = {term: len(posting) for term, posting in postings.items()}
doc_lengths = np.array([len(tokens) for tokens in doc_tokens], dtype=np.int32)
avg_doc_length = float(doc_lengths.mean())
print(f"Vocabulario stemmed: {len(postings):,}; longitud media: {avg_doc_length:.1f} tokens")


Vocabulario: 77,643; longitud media: 65.5 tokens


## 3. Jaccard binario

Cada documento y consulta se representa como conjunto de términos. La puntuación es la intersección dividida para la unión. Los documentos sin coincidencias reciben cero y no se priorizan frente a resultados con señal.


In [ ]:
def jaccard_scores(query):
    qset = set(tokenize(query))
    if not qset:
        return np.zeros(len(docs), dtype=np.float32)
    return np.fromiter((len(qset & terms) / len(qset | terms) if qset | terms else 0.0
                        for terms in doc_terms), dtype=np.float32, count=len(docs))


## 4. TF-IDF con similitud coseno

El vectorizador recibe los tokens ya normalizados. La matriz queda dispersa y con norma L2; el producto punto contra una consulta representa el coseno.


In [5]:
def identity(x):
    return x

tfidf = TfidfVectorizer(tokenizer=identity, preprocessor=identity, token_pattern=None,
                        lowercase=False, norm="l2", dtype=np.float32)
X_tfidf = tfidf.fit_transform(doc_tokens)

def tfidf_scores(query):
    q_tokens = tokenize(query)
    if not q_tokens:
        return np.zeros(len(docs), dtype=np.float32)
    q_vec = tfidf.transform([q_tokens])
    return (X_tfidf @ q_vec.T).toarray().ravel()


## 5. BM25

Implementación propia sobre el índice invertido. Valores iniciales: `k1=1.2`, `b=0.75`. La colección, el DF y las longitudes se mantienen fijos entre consultas.


In [ ]:
def bm25_scores(query, k1=1.2, b=0.75):
    scores = np.zeros(len(docs), dtype=np.float64)
    terms = set(tokenize(query))
    n_docs = len(docs)
    for term in terms:
        posting = postings.get(term)
        if not posting:
            continue
        term_df = len(posting)
        idf = math.log(1 + (n_docs - term_df + 0.5) / (term_df + 0.5))
        for i, freq in posting.items():
            norm = freq + k1 * (1 - b + b * doc_lengths[i] / avg_doc_length)
            scores[i] += idf * (freq * (k1 + 1)) / norm
    return scores

## 6. Embeddings con Sentence Transformers y ChromaDB

El modelo `sentence-transformers/multi-qa-MiniLM-L6-cos-v1` está orientado a búsqueda semántica y es relativamente compacto. La primera ejecución necesita internet para descargar pesos. Se generan vectores por lotes y se persisten localmente en `fiqa/chroma_db`; no subas esa carpeta al repositorio.


In [7]:
try:
    from sentence_transformers import SentenceTransformer
    import chromadb
    EMBEDDINGS_AVAILABLE = True
except ImportError as exc:
    EMBEDDINGS_AVAILABLE = False
    print(f"Embeddings desactivados: falta una dependencia opcional ({exc}).")

EMBED_MODEL = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
CHROMA_DIR = DATA_DIR / "chroma_db"
if EMBEDDINGS_AVAILABLE:
    MODEL = SentenceTransformer(EMBED_MODEL)
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = client.get_or_create_collection("fiqa_embeddings", metadata={"hnsw:space": "cosine"})
    if collection.count() != len(docs):
        try:
            client.delete_collection("fiqa_embeddings")
        except Exception:
            pass
        collection = client.create_collection("fiqa_embeddings", metadata={"hnsw:space": "cosine"})
        BATCH_SIZE = 256
        for start in range(0, len(docs), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(docs))
            vectors = MODEL.encode(doc_texts[start:end], batch_size=64, show_progress_bar=False,
                                   convert_to_numpy=True, normalize_embeddings=True)
            collection.add(ids=doc_ids[start:end], embeddings=vectors.tolist(),
                           documents=doc_texts[start:end])

def embedding_search(query, top_k=100):
    if not EMBEDDINGS_AVAILABLE:
        raise RuntimeError("Instala sentence-transformers y chromadb para activar embeddings.")
    vector = MODEL.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    result = collection.query(query_embeddings=[vector.tolist()], n_results=top_k)
    return [(doc_id, 1.0 - distance) for doc_id, distance in
            zip(result["ids"][0], result["distances"][0])]


c:\notebooks_IR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4206.11it/s]


## 7. Rankings y ejemplos

Para las métricas se comparan rankings hasta `RANK_CUTOFF`. Se muestran consulta, modelo, ID, etiqueta/fragmento y score; el ID es la clave para evaluar con qrels.


In [8]:
RANK_CUTOFF = 100

def rank_scores(scores, top_k=RANK_CUTOFF):
    # Desempate reproducible por ID del documento.
    order = np.lexsort((np.asarray(doc_ids), -np.asarray(scores)))[:top_k]
    return [(doc_ids[i], float(scores[i])) for i in order if scores[i] > 0]

def search_classic(model_name, query, top_k=RANK_CUTOFF):
    if model_name == "Jaccard":
        scores = jaccard_scores(query)
    elif model_name == "TF-IDF":
        scores = tfidf_scores(query)
    elif model_name == "BM25":
        scores = bm25_scores(query)
    else:
        raise ValueError(model_name)
    return rank_scores(scores, top_k)

def show_results(query, model_name, top_k=10):
    ranking = embedding_search(query, top_k) if model_name == "Embeddings" else search_classic(model_name, query, top_k)
    by_id = {doc_id: i for i, doc_id in enumerate(doc_ids)}
    return pd.DataFrame([{"posición": rank + 1, "id": doc_id, "etiqueta": doc_labels[by_id[doc_id]], "score": score}
                         for rank, (doc_id, score) in enumerate(ranking)])

# Ejemplo libre: cambia la consulta para inspeccionar los métodos disponibles.
consulta_demo = "How can I reduce the interest rate on my credit card debt?"
metodos_demo = ["Jaccard", "TF-IDF", "BM25"] + (["Embeddings"] if EMBEDDINGS_AVAILABLE else [])
for metodo in metodos_demo:
    print("\n", metodo)
    display(show_results(consulta_demo, metodo, top_k=5))



 Jaccard


,posición,id,etiqueta,score
0,1,360481,"""good vs """"bad"""" debt in the context of that p...",0.266667
1,2,317037,I would use student loans and avoid credit car...,0.250000
2,3,135112,As long as you can get a cash advance on the c...,0.222222
3,4,250815,"According to PayPal, if he or she used a credi...",0.200000
4,5,37449,You could buy some bitcoins with your credit c...,0.200000



 TF-IDF


,posición,id,etiqueta,score
0,1,360481,"""good vs """"bad"""" debt in the context of that p...",0.480118
1,2,215180,First and foremost you should do more research...,0.474909
2,3,345448,What makes a credit card risky is that it requ...,0.471974
3,4,264631,"""Transferring the balance of a credit card is ...",0.470961
4,5,432077,The post that you linked to saying that a pre-...,0.462983



 BM25


,posición,id,etiqueta,score
0,1,508510,If you hadn't done it already cut up the cards...,20.358117
1,2,360481,"""good vs """"bad"""" debt in the context of that p...",18.952588
2,3,200268,"""In the sole interest of improving your credit...",18.539475
3,4,571246,To confirm: you say you have credit card debt ...,17.782196
4,5,288268,1. you want /r/personalfiance 2. 1 payment or...,17.396672



 Embeddings


,posición,id,etiqueta,score
0,1,481898,"""The best way I know of to get the interest ra...",0.730836
1,2,547835,For those who are looking to improve credit fo...,0.678561
2,3,364361,"If you want to work within that constraint, th...",0.671993
3,4,559523,Get a loan at a decent interest rate and use t...,0.666766
4,5,220241,"First, make sure you have some money in a savi...",0.654233


## 8. Evaluación con qrels

Los qrels observados usan relevancia binaria (`score=1`). Se calculan Precision@10 y Recall@10 por consulta y MAP@100 del split. Se usa `dev` para desarrollar y comparar; después de fijar las decisiones, cambia `EVAL_SPLIT` a `test` y ejecuta una sola evaluación final. No uses `test` para ajustar parámetros.


In [9]:
def load_qrels(split):
    frame = pd.read_csv(QRELS_DIR / f"{split}.tsv", sep="\t")
    frame = frame[frame["score"] > 0]
    return {str(qid): set(group["corpus-id"].astype(str))
            for qid, group in frame.groupby("query-id")}

def precision_recall_ap(ranking, relevant, k=RANK_CUTOFF):
    ranked_ids = [doc_id for doc_id, _ in ranking[:k]]
    hits = [1 if doc_id in relevant else 0 for doc_id in ranked_ids]
    p10 = sum(hits[:10]) / 10
    r10 = sum(hits[:10]) / len(relevant) if relevant else 0.0
    hit_count = 0
    precision_sum = 0.0
    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            precision_sum += hit_count / rank
    ap_k = precision_sum / min(len(relevant), k) if relevant else 0.0
    return p10, r10, ap_k

def evaluate(split="dev", query_limit=None):
    qrels = load_qrels(split)
    qids = sorted(qrels, key=lambda x: int(x))
    if query_limit is not None:
        qids = qids[:query_limit]
    methods = ["Jaccard", "TF-IDF", "BM25"] + (["Embeddings"] if EMBEDDINGS_AVAILABLE else [])
    rows = []
    for qid in qids:
        query = queries.get(qid)
        if query is None:
            continue
        for method in methods:
            ranking = embedding_search(query, RANK_CUTOFF) if method == "Embeddings" else search_classic(method, query, RANK_CUTOFF)
            p10, r10, ap = precision_recall_ap(ranking, qrels[qid])
            rows.append({"query_id": qid, "modelo": method, "P@10": p10, "R@10": r10, "AP@100": ap})
    per_query = pd.DataFrame(rows)
    summary = per_query.groupby("modelo")[["P@10", "R@10", "AP@100"]].mean().sort_values("AP@100", ascending=False)
    return per_query, summary

EVAL_SPLIT = "dev"
metricas_por_consulta, resumen = evaluate(EVAL_SPLIT)
display(resumen)


,P@10,R@10,AP@100
modelo,,,
Embeddings,0.0994,0.440286,0.323811
BM25,0.0586,0.286289,0.197307
TF-IDF,0.0474,0.223118,0.133262
Jaccard,0.0122,0.061456,0.035069


## 9. Análisis, interfaz y entregables

1. Compara P@10, R@10 y MAP@100; revisa consultas concretas donde embeddings cambian el ranking y contrasta esos casos con BM25/TF-IDF.
2. Fija modelo, parámetros y preprocesamiento según `dev`; registra decisiones y limitaciones. Luego evalúa `test` una sola vez.
3. Convierte `show_results` y los recuperadores en una interfaz Streamlit: caja de consulta, selector de modelo, top-n, resultados y métricas. La base persistente debe reconstruirse o incluir instrucciones claras para generarla.
4. Entrega URL desplegada, código fuente con README e informe técnico de máximo cinco páginas con descripción de FiQA, diseño, consultas de ejemplo y análisis de métricas.

El notebook no despliega por sí mismo la aplicación; Streamlit Community Cloud es una opción gratuita, pero la carga del modelo y la disponibilidad de los archivos deben configurarse para el entorno de despliegue.
